Vector Store Retriever


In [1]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings  
from langchain_community.vectorstores import FAISS

# Create 5 Document objects
docs = [
    Document(page_content="Python is a programming language", metadata={"id": 1}),
    Document(page_content="Java is also a programming language", metadata={"id": 2}),
    Document(page_content="Cats sleep often during the day", metadata={"id": 3}),
    Document(page_content="Dogs bark at strangers", metadata={"id": 4}),
    Document(page_content="Birds fly in the sky", metadata={"id": 5}),
]

# Initialize embedding model
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Build vector store
vs = FAISS.from_documents(docs, embedding=emb)

# Create retriever
retriever = vs.as_retriever(search_kwargs={"k": 3})

# Query
query = "Which animals are domestic pets?"
out_docs = retriever.invoke(query)

print("Retriever results:")
for d in out_docs:
    print(f"ID: {d.metadata.get('id')}, Content: {d.page_content}")

Retriever results:
ID: 4, Content: Dogs bark at strangers
ID: 5, Content: Birds fly in the sky
ID: 3, Content: Cats sleep often during the day


In [2]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

load_dotenv()

# Create 5 Document objects
docs = [
    Document(page_content="Python is a programming language", metadata={"id": 1}),
    Document(page_content="Java is also a programming language", metadata={"id": 2}),
    Document(page_content="Cats sleep often during the day", metadata={"id": 3}),
    Document(page_content="Dogs bark at strangers", metadata={"id": 4}),
    Document(page_content="Birds fly in the sky", metadata={"id": 5}),
]

# Initialize embedding model
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Build vector store
vs = FAISS.from_documents(docs, embedding=emb)

# Create retriever with different search types
print("=" * 60)
print("FAISS Retriever Examples")
print("=" * 60)

# 1. Similarity retriever (default)
retriever_similarity = vs.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# 2. MMR retriever (Max Marginal Relevance - for diversity)
retriever_mmr = vs.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 5, "lambda_mult": 0.5}
)

# 3. Similarity score threshold retriever
retriever_threshold = vs.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.5, "k": 3}
)

# Query
query = "Which animals are domestic pets?"
print(f"\nQuery: '{query}'")
print("-" * 60)

# Test similarity retriever
print("\n1. Similarity Search Results (k=3):")
out_docs = retriever_similarity.invoke(query)
for i, d in enumerate(out_docs, 1):
    print(f"   {i}. ID: {d.metadata.get('id')}, Content: {d.page_content}")

# Test MMR retriever
print("\n2. MMR Search Results (diverse results):")
out_docs_mmr = retriever_mmr.invoke(query)
for i, d in enumerate(out_docs_mmr, 1):
    print(f"   {i}. ID: {d.metadata.get('id')}, Content: {d.page_content}")

# Test similarity search with scores
print("\n3. Similarity Search with Scores:")
results_with_scores = vs.similarity_search_with_score(query, k=3)
for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"   {i}. ID: {doc.metadata.get('id')}, Score: {score:.4f}, Content: {doc.page_content}")

# Search for programming languages
print("\n" + "=" * 60)
query2 = "programming languages"
print(f"Query: '{query2}'")
print("-" * 60)

results2 = vs.similarity_search(query2, k=2)
for i, d in enumerate(results2, 1):
    print(f"{i}. ID: {d.metadata.get('id')}, Content: {d.page_content}")

# Search for animals
print("\n" + "=" * 60)
query3 = "animals"
print(f"Query: '{query3}'")
print("-" * 60)

results3 = vs.similarity_search(query3, k=3)
for i, d in enumerate(results3, 1):
    print(f"{i}. ID: {d.metadata.get('id')}, Content: {d.page_content}")

# Save and load FAISS index (optional)
print("\n" + "=" * 60)
print("Saving and Loading FAISS Index")
print("-" * 60)

# Save
save_path = "faiss_animal_index"
vs.save_local(save_path)
print(f"Saved to: {save_path}")

# Load
loaded_vs = FAISS.load_local(save_path, emb, allow_dangerous_deserialization=True)
print(f"Loaded successfully. Documents: {len(loaded_vs.docstore._dict)}")

# Verify loaded retriever
loaded_retriever = loaded_vs.as_retriever(search_kwargs={"k": 2})
test_query = "cats"
test_results = loaded_retriever.invoke(test_query)
print(f"\nTest loaded retriever with query '{test_query}':")
for i, d in enumerate(test_results, 1):
    print(f"   {i}. {d.page_content}")

print("\n" + "=" * 60)
print("All operations completed successfully")
print("=" * 60)

FAISS Retriever Examples

Query: 'Which animals are domestic pets?'
------------------------------------------------------------

1. Similarity Search Results (k=3):
   1. ID: 4, Content: Dogs bark at strangers
   2. ID: 5, Content: Birds fly in the sky
   3. ID: 3, Content: Cats sleep often during the day

2. MMR Search Results (diverse results):
   1. ID: 4, Content: Dogs bark at strangers
   2. ID: 1, Content: Python is a programming language
   3. ID: 3, Content: Cats sleep often during the day

3. Similarity Search with Scores:
   1. ID: 4, Score: 0.7821, Content: Dogs bark at strangers
   2. ID: 5, Score: 0.7880, Content: Birds fly in the sky
   3. ID: 3, Score: 0.7971, Content: Cats sleep often during the day

Query: 'programming languages'
------------------------------------------------------------
1. ID: 1, Content: Python is a programming language
2. ID: 2, Content: Java is also a programming language

Query: 'animals'
--------------------------------------------------------

BM25 Retriever


In [3]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# إنشاء 5 مستندات بسيطة
docs = [
    Document(page_content="The sky is blue during the day", metadata={"id": 1}),
    Document(page_content="At night the stars light up the sky", metadata={"id": 2}),
    Document(page_content="Blue whales are the largest animals on Earth", metadata={"id": 3}),
    Document(page_content="Birds can fly high up in the sky", metadata={"id": 4}),
    Document(page_content="Deep sea creatures live in darkness", metadata={"id": 5}),
]

# إنشاء BM25 retriever (يبحث بالكلمات وليس بالمعنى)
retriever = BM25Retriever.from_documents(docs, k=3)

# البحث
query = "sky blue animals"
results = retriever.invoke(query)

# عرض النتائج
print("Query:", query)
print("-" * 40)
for doc in results:
    print(f"ID: {doc.metadata['id']} | {doc.page_content}")

Query: sky blue animals
----------------------------------------
ID: 1 | The sky is blue during the day
ID: 3 | Blue whales are the largest animals on Earth
ID: 4 | Birds can fly high up in the sky


Ensemble / Hybrid Retriever

In [1]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import warnings

load_dotenv()

# 5 documents
docs = [
    Document(page_content="AI research is growing fast", metadata={"id": 1}),
    Document(page_content="Machine learning models perform many tasks", metadata={"id": 2}),
    Document(page_content="Cooking and baking are arts", metadata={"id": 3}),
    Document(page_content="Dogs are loyal animals", metadata={"id": 4}),
    Document(page_content="Software engineering is a discipline", metadata={"id": 5}),
]

# BM25 retriever
bm25 = BM25Retriever.from_documents(docs, k=2)

# Vector retriever
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vs = FAISS.from_documents(docs, embedding=emb)
vector_retriever = vs.as_retriever(search_kwargs={"k": 2})

# محاولة استيراد الـ retriever المناسب
try:
    # المحاولة الأولى: الطريقة الجديدة (MergeRetriever)
    from langchain.retrievers.merger import MergeRetriever
    hybrid_retriever = MergeRetriever(
        retrievers=[bm25, vector_retriever], 
        weights=[0.5, 0.5]
    )
    print("✅ Using MergeRetriever (new)")
    
except ImportError:
    try:
        # المحاولة الثانية: الطريقة القديمة (EnsembleRetriever)
        from langchain.retrievers import EnsembleRetriever
        hybrid_retriever = EnsembleRetriever(
            retrievers=[bm25, vector_retriever], 
            weights=[0.5, 0.5]
        )
        print("✅ Using EnsembleRetriever (old)")
        
    except ImportError:
        # المحاولة الثالثة: طريقة يدوية
        print("⚠️ No hybrid retriever found, using manual combination")
        
        class ManualHybridRetriever:
            def __init__(self, retrievers, weights):
                self.retrievers = retrievers
                self.weights = weights
            
            def invoke(self, query):
                from collections import defaultdict
                
                # جمع النتائج من جميع الباحثين
                all_results = []
                for retriever in self.retrievers:
                    all_results.extend(retriever.invoke(query))
                
                # إزالة التكرارات (حسب ID)
                seen_ids = set()
                unique_results = []
                for doc in all_results:
                    if doc.metadata['id'] not in seen_ids:
                        unique_results.append(doc)
                        seen_ids.add(doc.metadata['id'])
                
                return unique_results[:2]  # خذ أفضل 2
        
        hybrid_retriever = ManualHybridRetriever(
            retrievers=[bm25, vector_retriever],
            weights=[0.5, 0.5]
        )

# Query
query = "what are models in AI"
results = hybrid_retriever.invoke(query)

print("\n" + "=" * 60)
print("Hybrid Retriever Results")
print("=" * 60)
print(f"Query: '{query}'")
print("-" * 50)

for i, d in enumerate(results, 1):
    print(f"{i}. ID {d.metadata.get('id')}: {d.page_content}")

print("-" * 50)
print(f"عدد النتائج: {len(results)}")

⚠️ No hybrid retriever found, using manual combination

Hybrid Retriever Results
Query: 'what are models in AI'
--------------------------------------------------
1. ID 1: AI research is growing fast
2. ID 2: Machine learning models perform many tasks
--------------------------------------------------
عدد النتائج: 2
